In [292]:
import pandas as pd
import os
import numpy as np
import re

In [293]:
pd.set_option('display.max_columns', None)

# Data curation

## **STEP 1**. Merge dicomtocsv_series.csv

### <span style="color:blue">**Main**</span> (determine which part of data)

In [ ]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/R3Data"
file_series_p1 = "dicomtocsv_series_20250527.xlsx"
file_series_p2 = "dicomtocsv_series_20260803.xlsx"
file_series_p3 = "dicomtocsv_series_20260812.xlsx"

In [ ]:
df_dicom_series_p1 = pd.read_excel(os.path.join(file_path, file_series_p1))
df_dicom_series_p2 = pd.read_excel(os.path.join(file_path, file_series_p2))
df_dicom_series_p3 = pd.read_excel(os.path.join(file_path, file_series_p3))

In [ ]:
df_dicom_series_p1.shape, df_dicom_series_p2.shape, df_dicom_series_p3.shape

In [ ]:
df_tmp = pd.concat((df_dicom_series_p1, df_dicom_series_p2), axis=0)
df_all = pd.concat((df_tmp, df_dicom_series_p3), axis=0)

# df_all = pd.concat((df_dicom_series_p2, df_dicom_series_p3), axis=0)

df_all.shape

(3136, 7)

In [298]:
df_all['StudyDate'] = pd.to_datetime(df_all['StudyDate'], format='%Y%m%d').dt.strftime('%Y-%m-%d')

In [299]:
df_all.head(5)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath
0,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
3,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
4,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73500000,L CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


## **STEP 2**. Group by PatientID

### <span style="color:blue">**Main**</span>

In [300]:
df_sort = df_all.sort_values(
        by=['PatientID', 'StudyDate', 'AccessionNumber'],
        ignore_index=True
    )

num_patient = df_sort["PatientID"].nunique()
print(num_patient)

536


In [301]:
df_sort.head(5)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath
0,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
3,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
4,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73500000,L CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


In [302]:
df_step2 = df_sort.copy()

## **STEP 2.1**. Calculate Age @ Study <span style="color:red">(OPTIONAL)</span>
### <span style="color:red"> New batches: Missing patient/_demo birth_date info. 50 patients; All: 54 patients </span>

### <span style="color:darkcyan">**Function**</span>

In [303]:
from pandas import Int64Dtype
def calculate_age_at_study(df):
    """
    Calculates the patient's age in years at the time of the study 
    based on the 'PatientBirthDate' and 'StudyDate' columns.
    """
    
    # 1. Ensure date columns are in datetime format
    # The format 'YYYY-MM-DD' is used for parsing
    try:
        df['BIRTH_DATE_DT'] = pd.to_datetime(df['PatientBirthDate'], format='%Y-%m-%d')
        # Note: The StudyDate column often includes time (e.g., '2020-06-01 09:10:52').
        # We can let pandas infer the format for this one since it's cleaner.
        df['StudyDate_DT'] = pd.to_datetime(df['StudyDate'], errors='coerce') 
    except ValueError as e:
        print(f"Error parsing date format: {e}. Please check your date column formats.")
        return df

    # 2. Calculate the difference in days
    time_difference = df['StudyDate_DT'] - df['BIRTH_DATE_DT']
    
    # 3. Convert the difference into whole years (integer format)
    # The .dt.days attribute gives the number of days, which is divided by 365.25 
    # and then explicitly cast to an integer to capture only the full years elapsed.
    mask = df['BIRTH_DATE_DT'].notna() & df['StudyDate_DT'].notna()
    df.loc[mask, 'PatientAge'] = (time_difference[mask].dt.days / 365.25).astype(int)
    # df['PatientAge'] = (time_difference.dt.days / 365.25).astype('Int64')
    
    # Clean up the intermediate columns
    
    df = df.drop(columns=['BIRTH_DATE_DT', 'StudyDate_DT'])
    
    return df

### <span style="color:blue">**Main**</span>

In [304]:
demo_file_path_cancer = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/Cancer/Cleaned/patient_demo.xlsx"
demo_file_path_control = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/Control/Cleaned/patient_demo.xlsx"

demo_cancer = pd.read_excel(demo_file_path_cancer)[["PATIENT_STUDY_ID", "BIRTH_DATE"]]
demo_control = pd.read_excel(demo_file_path_control)[["PATIENT_STUDY_ID", "BIRTH_DATE"]]

In [305]:
demo =  pd.concat((demo_cancer, demo_control), axis=0)
demo = demo.drop_duplicates(subset=demo.columns.tolist(), keep = 'first').reset_index(drop = True)

In [306]:
demo.rename(columns={'PATIENT_STUDY_ID': 'PatientID', 'BIRTH_DATE': 'PatientBirthDate'}, inplace=True)

In [307]:
demo.head(3)

,PatientID,PatientBirthDate
0,4330018595,1965-07-01
1,4330029102,1974-07-01
2,4330044371,1975-07-01


In [308]:
df_birth = pd.merge(df_sort, demo, on='PatientID', how='left')

In [309]:
df_sort.shape, demo.shape, df_birth.shape

((3136, 7), (47245, 2), (3136, 8))

In [310]:
df_step2 = calculate_age_at_study(df_birth)

In [311]:
df_step2['PatientID'].nunique(), df_step2.loc[df_step2['PatientAge'].notna(), 'PatientID'].nunique(), df_step2.loc[df_step2['PatientAge'].isna(), 'PatientID'].nunique()

(536, 486, 50)

In [312]:
df_step2.loc[df_step2['PatientAge'].isna(), 'PatientID'].unique()

array([4333000921, 4333012455, 4333016761, 4333024138, 4333027901,
       4333031047, 4333032244, 4333042111, 4333043117, 4333055254,
       4333057495, 4333073168, 4333078808, 4333082262, 4333089213,
       4333098676, 4333322368, 4333324955, 4333339192, 4333339578,
       4333343566, 4333369221, 4333370187, 4333391549, 4333398550,
       4333402040, 4333419154, 4333447879, 4333447971, 4333455214,
       4333471743, 4333481825, 4333483028, 4333490388, 4333493561,
       4333520281, 4333520420, 4333532845, 4333535298, 4333584454,
       4333588635, 4333591910, 4333596430, 4333606098, 4333643575,
       4333646458, 4333651350, 4333659749, 4333661051, 4333667640],
      dtype=int64)

In [313]:
df_step2

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath,PatientBirthDate,PatientAge
0,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0
1,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0
2,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0
3,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0
4,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73500000,L CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0
...,...,...,...,...,...,...,...,...,...
3131,4333669308,2022-06-07,SCREENING MAMMOGRAM WITH TOMO,453037184,73500000,R MLO Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...,1957-07-01,64.0
3132,4333669687,2021-10-13,SCREEN MAMMO WITH TOMO BILATERAL,454228662,73500000,L CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...,1953-07-01,68.0
3133,4333669687,2021-10-13,SCREEN MAMMO WITH TOMO BILATERAL,454228662,73500000,L MLO Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...,1953-07-01,68.0
3134,4333669687,2021-10-13,SCREEN MAMMO WITH TOMO BILATERAL,454228662,73500000,R CC Breast Tomosynthesis Image Slabs,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...,1953-07-01,68.0


## **STEP 3.** Add tags (Study, Side, Series)

In [314]:
dicom = df_step2

In [315]:
dicom.head(3)

,PatientID,StudyDate,StudyDescription,AccessionNumber,SeriesNumber,SeriesDescription,FolderPath,PatientBirthDate,PatientAge
0,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0
1,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,L MLO Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0
2,4333000414,2021-06-12,SCREENING MAMMO WITH TOMO,455560367,73200000,R CC Breast Tomosynthesis Image,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...,1960-07-01,60.0


In [316]:
dicom_copy = dicom.copy()

### <span style="color:blue"> **Study**</span> (SCREEN, DIAG)

In [317]:
study_types = {
    "DIAG":   ["DIAG", "DIAGNOSTIC", "DX"],
    "SCREEN": ["SCREENING", "SCREEN"],
}

In [318]:
column_to_check = 'StudyDescription'
type_column = 'Study'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in study_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Side**</span> (R, L)

In [319]:
side_types = {
    "R": ["RIGHT", "RT", "R XCCL", "R MLO", "R CC", "R ML", "R SIO", "R LM"],
    "L": ["LEFT", "LT", "L XCCL", "L MLO", "L CC", "L ML", "L SIO", "L LM"],
}

In [320]:
column_to_check = 'SeriesDescription'
type_column = 'Side'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in side_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Series**</span> (DBT, IN2D, C VIEW, SECURE)

In [321]:
series_types = {
    "DBT":    ["Breast Tomosynthesis"],
    "IN2D":   ["Intelligent 2D"],
    "C VIEW": ["C-View"],
    "SECURE": ["SecurView", "CAD SC"],
}

ffdm_exact = ["R CC", "R MLO", "R ML", "L CC", "L MLO", "L ML",
              "L XCCL", "R XCCL", "R LM", "L LM",
              "R SIO", "L SIO", "RT", "LT"
              ]

In [322]:
column_to_check = 'SeriesDescription'
type_column = 'Series'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in series_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

# Exact matches for FFDM
ffdm_mask = dicom_copy[column_to_check].str.strip().str.upper().isin(
    [t.upper() for t in ffdm_exact]
)
dicom_copy.loc[ffdm_mask, type_column] = "FFDM"

### <span style="color:blue">**View**</span> (MLO, CC)

In [323]:
view_types = {
    "MLO": ["L MLO", "R MLO"],
    "CC":  ["L CC", "R CC"],
    "ML":  ["L ML", "R ML"],
    "XCCL":  ["L XCCL", "R XCCL"],
    "LM":  ["L LM", "R LM"],
    "SIO": ["R SIO", "L SIO"],
}

In [324]:
column_to_check = 'SeriesDescription'
type_column = 'View'

dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

for label, terms in view_types.items():
    pattern = '|'.join(re.escape(t) for t in terms)
    mask = dicom_copy[column_to_check].str.contains(pattern, case=False, na=False, regex=True)
    dicom_copy.loc[mask, type_column] = label

### <span style="color:blue">**Reorder columns**</span>

In [325]:
dicom_copy.columns

Index(['PatientID', 'StudyDate', 'StudyDescription', 'AccessionNumber',
       'SeriesNumber', 'SeriesDescription', 'FolderPath', 'PatientBirthDate',
       'PatientAge', 'Study', 'Side', 'Series', 'View'],
      dtype='object')

In [326]:
# dicom_copy = dicom_copy[['PatientID', 
#         'AccessionNumber', 'Study', 'Side', 'Series', 'View',    
#         'StudyDate', 'StudyDescription',
#         'SeriesDescription', 'SeriesNumber', 
#         'FolderPath'
#         ]]

dicom_copy = dicom_copy[['PatientID', 'PatientBirthDate', 'PatientAge',
        'AccessionNumber', 'StudyDate',
        'Study', 'Side', 'Series', 'View',    
        'StudyDescription',
        'SeriesDescription', 'SeriesNumber', 
        'FolderPath'
        ]]

In [327]:
dicom_copy.head(3)

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,CC,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,ML,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,R,DBT,CC,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


### <span style="color:#FF6347;">**SAVE**</span> file

In [328]:
path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation"

In [ ]:
output_file = os.path.join(path,'dicom_tag_v2' + ".xlsx")
dicom_copy.to_excel(output_file, index=False)

### <span style="color:#FF6347;">**READ**</span> file

In [ ]:
file_path = os.path.join(path,'dicom_tag_v2' + ".xlsx")
dicom = pd.read_excel(file_path)

In [331]:
dicom[dicom["Series"]=="DBT"]

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,CC,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,ML,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,R,DBT,CC,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
3,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,R,DBT,ML,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
4,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,CC,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3131,4333669308,1957-07-01,64.0,453037184,2022-06-07,SCREEN,R,DBT,ML,SCREENING MAMMOGRAM WITH TOMO,R MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
3132,4333669687,1953-07-01,68.0,454228662,2021-10-13,SCREEN,L,DBT,CC,SCREEN MAMMO WITH TOMO BILATERAL,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
3133,4333669687,1953-07-01,68.0,454228662,2021-10-13,SCREEN,L,DBT,ML,SCREEN MAMMO WITH TOMO BILATERAL,L MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...
3134,4333669687,1953-07-01,68.0,454228662,2021-10-13,SCREEN,R,DBT,CC,SCREEN MAMMO WITH TOMO BILATERAL,R CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\376GB_Tomos_2nd_600\Pat...


In [332]:
dicom[dicom["Series"]=="DBT"]["PatientID"].unique().size

536

In [333]:
dicom[dicom['PatientID']==4333000414]

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,CC,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
1,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,ML,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
2,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,R,DBT,CC,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
3,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,R,DBT,ML,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
4,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,CC,SCREENING MAMMO WITH TOMO,L CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
5,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,L,DBT,ML,SCREENING MAMMO WITH TOMO,L MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
6,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,R,DBT,CC,SCREENING MAMMO WITH TOMO,R CC Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...
7,4333000414,1960-07-01,60.0,455560367,2021-06-12,SCREEN,R,DBT,ML,SCREENING MAMMO WITH TOMO,R MLO Breast Tomosynthesis Image Slabs,73500000,J:\Testing\jlee\MO_DBT\166GB_Tomos_First100\Pa...


In [334]:
dicom[dicom['PatientID']==4330018595]

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
